In [ ]:
import serial
import time
import csv

arduino = serial.Serial("COM7", 115200, timeout=1)

time.sleep(2)

print("Arduino connected!")
print("Recording...")

data = []

start = time.time()

while time.time() - start < 10:

    line = arduino.readline().decode().strip()

    if line:
        value = int(line)

        timestamp = time.time() - start

        data.append([timestamp, value])

        print(timestamp, value)

arduino.close()

# Save CSV
with open("eog_data.csv", "w", newline="") as file:

    writer = csv.writer(file)

    writer.writerow(["Timestamp", "EOG"])

    writer.writerows(data)

print("Recording finished!")
print("Saved as eog_data.csv")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# CSV file name
file = "eog_data.csv"

# Read CSV
data = pd.read_csv(file)

# Get the last 10 seconds
#last_time = data["Timestamp"].max()

#start_time = last_time - 10

#last_10 = data[data["Timestamp"] >= start_time]

print(len(data))
plt.plot(data["EOG"])
plt.show()

"""
# Plot
plt.figure(figsize=(12, 5))

plt.plot(
    last_10["Timestamp"],
    last_10["EOG"]
)

plt.xlabel("Time (seconds)")
plt.ylabel("EOG Value")
plt.title("EOG Signal - Last 10 Seconds")

plt.grid(True)
plt.tight_layout()

# Save plot
plt.savefig(
    "EOG_last_10_seconds.png",
    dpi=300
)

# Show plot
plt.show()

print("Plot saved as EOG_last_10_seconds.png")
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch


# ==========================================
# LOAD DATA
# ==========================================

data = pd.read_csv("eog_data.csv")

time = data["Timestamp"].values
eog = data["EOG"].values

# ==========================================
# GET LAST 10 SECONDS
# ==========================================

last_time = time[-1]

mask = time >= last_time - 10

time = time[mask]
eog = eog[mask]

# Reset time to 0–10 seconds
time = time - time[0]

# ==========================================
# SAMPLING RATE
# ==========================================

fs = len(eog) / (time[-1] - time[0])

print("Samples:", len(eog))
print("Duration:", time[-1], "seconds")
print("Sampling rate:", fs, "Hz")

# ==========================================
# REMOVE DC OFFSET
# ==========================================

eog_centered = eog - np.mean(eog)

# ==========================================
# BANDPASS FILTER
# ==========================================

lowcut = 0.1
highcut = 10

nyquist = fs / 2

b, a = butter(
    4,
    [lowcut / nyquist, highcut / nyquist],
    btype="band"
)

eog_bandpass = filtfilt(
    b,
    a,
    eog_centered
)

# ==========================================
# 50 Hz NOTCH FILTER
# ==========================================

notch_freq = 50
Q = 30

b_notch, a_notch = iirnotch(
    notch_freq,
    Q,
    fs
)

eog_filtered = filtfilt(
    b_notch,
    a_notch,
    eog_bandpass
)

# ==========================================
# PLOT
# ==========================================

plt.figure(figsize=(14, 5))

plt.plot(eog_filtered)


plt.ylabel("EOG Amplitude")
plt.title("Cleaned EOG Signal - Last 10 Seconds")

plt.grid(True)
plt.tight_layout()

plt.show()